In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np

In [2]:
import tensorflow as tf

# Load the CIFAR-10 dataset
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

# Display the shapes of the loaded data to confirm
print(f"Shape of training images: {train_images.shape}")
print(f"Shape of training labels: {train_labels.shape}")
print(f"Shape of testing images: {test_images.shape}")
print(f"Shape of testing labels: {test_labels.shape}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
Shape of training images: (50000, 32, 32, 3)
Shape of training labels: (50000, 1)
Shape of testing images: (10000, 32, 32, 3)
Shape of testing labels: (10000, 1)


In [3]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 1


In [2]:
import tensorflow as tf

(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.cifar10.load_data()

num_classes = 10

def preprocess_image(image, label):
    image = tf.image.resize(image, (224, 224)) # Resize images
    image = tf.cast(image, tf.float32) / 255.0 # Normalize
    return image, label

# Create tf.data.Dataset objects
batch_size = 32 # You can adjust this batch size
train_dataset = tf.data.Dataset.from_tensor_slices((x_train_raw, y_train))
train_dataset = train_dataset.map(preprocess_image).batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test_raw, y_test))
test_dataset = test_dataset.map(preprocess_image).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Note: x_train and x_test are now tf.data.Dataset objects, not numpy arrays
# If you need them as numpy arrays for a specific reason, you'd iterate through the dataset
# For training, you can directly pass these datasets to model.fit()

In [11]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Explicitly build the model to ensure all variables are created before fitting
model.build(input_shape=(None, 224, 224, 3)) # None for batch size

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

In [14]:
history = model.fit(
    train_dataset, # Use the preprocessed tf.data.Dataset
    epochs=20,
    validation_data=test_dataset, # Use the preprocessed test dataset for validation
    callbacks=[early_stop, lr_schedule]
)

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 86s 55ms/step - accuracy: 0.3677 - loss: 1.7574 - val_accuracy: 0.5354 - val_loss: 1.3041 - learning_rate: 0.0010
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 137s 52ms/step - accuracy: 0.5614 - loss: 1.2426 - val_accuracy: 0.5766 - val_loss: 1.2019 - learning_rate: 0.0010
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 51ms/step - accuracy: 0.6246 - loss: 1.0704 - val_accuracy: 0.5983 - val_loss: 1.1385 - learning_rate: 0.0010
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 51ms/step - accuracy: 0.6787 - loss: 0.9266 - val_accuracy: 0.5974 - val_loss: 1.1724 - learning_rate: 0.0010
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 79s 51ms/step - accuracy: 0.7147 - loss: 0.8155 - val_accuracy: 0.5846 - val_loss: 1.3095 - learning_rate: 0.0010
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 51ms/step - accuracy: 0.7532 - loss: 0.7098 - val_accuracy: 0.5863 - val_loss: 1.4298 - learning_rate: 0.0010
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 79s 51ms/step - accur

In [17]:
model.trainable = True # Ensure all layers are initially trainable

# Freeze the convolutional base layers (0 to 5)
# The model has 9 layers in total: 3 Conv2D, 3 MaxPooling2D, Flatten, 2 Dense.
# Freezing layers up to index 6 (excluding 6) means freezing the Conv2D and MaxPooling2D layers.
for layer in model.layers[:6]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Corrected to tf.keras
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_dataset, # Use the preprocessed tf.data.Dataset
    epochs=20,
    validation_data=test_dataset, # Use the preprocessed test dataset for validation
    callbacks=[early_stop, lr_schedule]
)

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - accuracy: 0.6794 - loss: 0.9227 - val_accuracy: 0.6067 - val_loss: 1.1160 - learning_rate: 1.0000e-05
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 42s 27ms/step - accuracy: 0.6864 - loss: 0.8961 - val_accuracy: 0.6098 - val_loss: 1.1097 - learning_rate: 1.0000e-05
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 38s 24ms/step - accuracy: 0.6898 - loss: 0.8839 - val_accuracy: 0.6095 - val_loss: 1.1053 - learning_rate: 1.0000e-05
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - accuracy: 0.6932 - loss: 0.8750 - val_accuracy: 0.6118 - val_loss: 1.1017 - learning_rate: 1.0000e-05
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - accuracy: 0.6954 - loss: 0.8678 - val_accuracy: 0.6122 - val_loss: 1.0989 - learning_rate: 1.0000e-05
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - accuracy: 0.6977 - loss: 0.8617 - val_accuracy: 0.6126 - val_loss: 1.0965 - learning_rate: 1.0000e-05
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━

In [19]:
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=2)

print("Test Accuracy:", test_accuracy)

313/313 - 7s - 21ms/step - accuracy: 0.6199 - loss: 1.0814
Test Accuracy: 0.6198999881744385
